## Prepare Dynamic Wflow Initial Condition Startup State

In [1]:
import os
import warnings
from pathlib import Path
import sys

os.environ.pop("DEBUG", None)
os.environ.setdefault("MPLCONFIGDIR", "/tmp/flood-rm-matplotlib")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
warnings.filterwarnings("ignore")

from study_location import bootstrap
session = bootstrap()
location_root = session.location_root
repo_root = session.repo_root
for module_name in list(sys.modules):
    if module_name.split(".", 1)[0] in {"design_events", "sfincs_runs", "wflow_runs"}:
        sys.modules.pop(module_name, None)

import pandas as pd
import yaml
from IPython.display import display

# source inputs for Wflow warmup and event forcing.
from collect_sources import collect_warmup
# event selection, SFINCS staging, and handoff readiness.
from sfincs_runs.scenarios import plan_example
# standard paths and readiness tables.
from wflow_runs.notebook import load_runtime
# meteo forcing, discharge handoff, and acceptance gates.
from wflow_runs import (
    build_meteo,
    plan_handoff,
    plan_streamflow,
    plan_warmup,
    prepare_instates,
    prepare_handoff,
    validate_geometry,
    validate_warmup_forcing,
    validate_staticmaps,
    validate_wflow_sfincs_handoff_artifacts_current,
    validate_instates,
    validate_wflow_reservoir_states,
    repair_wflow_staticmaps_nodata,
    write_wflow_reservoir_readiness,
)

runtime = load_runtime(Path("../..").resolve(), wflow_domain_review_required=False)
location_root = runtime.location_root
repo_root = runtime.repo_root
config = runtime.config
paths = runtime.paths


### Select Event and Plan Handoff


In [2]:
EVENT_ID = None  # None -> highest-return-period catalog event; or e.g. "design_0369"
CATALOG_PATH = "data/event_catalog/catalog/probability_catalog.csv"

example_plan = plan_example(
    config,
    {"location_root": location_root},
    catalog_path=CATALOG_PATH,
    event_id=EVENT_ID,
)
display(example_plan)

if example_plan.event_id in (None, ""):
    raise RuntimeError(
        "No dynamic handoff event_id could be selected. "
        f"plan_example status={example_plan.status}; issues={example_plan.issues}"
    )

event_id = str(example_plan.event_id)
catalog = pd.read_csv(location_root / CATALOG_PATH, dtype={"event_id": str})
catalog["event_id"] = catalog["event_id"].astype(str).str.strip()
selected_rows = catalog[catalog["event_id"] == event_id]
if selected_rows.empty:
    preview = ", ".join(catalog["event_id"].head(8).astype(str))
    raise RuntimeError(f"Selected event_id {event_id!r} is not in {location_root / CATALOG_PATH}. First catalog IDs: {preview}")
selected = selected_rows.iloc[0]

handoff_plan = plan_handoff(config, location_root, event_id, catalog_path=location_root / CATALOG_PATH)
display(handoff_plan)

streamflow_realization = plan_streamflow(config, location_root, event_id, catalog_path=location_root / CATALOG_PATH)
display(streamflow_realization)
if streamflow_realization["status"].eq("failed").any():
    raise RuntimeError("Selected event is not ready for rainfall-driven Wflow generation (missing rainfall member or amplification reference gage).")


InlandCoupledExamplePlan(status='ready', event_id='design_0499', event_reference_time='1998-10-17 17:00:00', catalog_path='/home/grahamhults/projects/Flood-RM/locations/austin/data/event_catalog/catalog/probability_catalog.csv', handoff_path='/home/grahamhults/projects/Flood-RM/locations/austin/data/wflow/domain_set_handoff.yaml', wflow_event_dir='data/wflow/events/design_0499', wflow_discharge_forcing='data/wflow/events/design_0499/sfincs_discharge.nc', sfincs_scenario_dir='/home/grahamhults/projects/Flood-RM/locations/austin/data/sfincs/scenarios/design_0499/austin_p5u', forcing_manifest='/home/grahamhults/projects/Flood-RM/locations/austin/data/sfincs/scenarios/design_0499/austin_p5u/forcing_manifest.json', stage_command='stage_scenarios(runtime_config, {"location_root": location_root}, catalog_path="/home/grahamhults/projects/Flood-RM/locations/austin/data/event_catalog/catalog/probability_catalog.csv", event_ids=[\'design_0499\'], force=True)', sfincs_dry_run_command='uv run pytho

event_id                                                            design_0499
window_start                                                1998-10-15T17:00:00
window_end                                                  1998-10-20T17:00:00
discharge_source                                                  wflow_dynamic
state_policy                                                    shared_baseline
baseline_id                                                        baseline_90d
warmup_days                                                                90.0
warmup_baseline_root          /home/grahamhults/projects/Flood-RM/locations/...
sfincs_discharge_forcing      /home/grahamhults/projects/Flood-RM/locations/...
dynamic_handoff_acceptance    /home/grahamhults/projects/Flood-RM/locations/...
acceptance_status                                                       missing
Name: dynamic_wflow_handoff_plan, dtype: object

,check,status,message
0,catalog_rainfall_member,passed,member=rainfall_austin_72h_rank0003; scale=1.4...
1,amplification_reference_gage,passed,primary_reference_gage=08155400


### Rerun Control


In [3]:
rerun = True


### Verify Wflow Source Geometry and Static Maps


In [4]:
river_geometry = location_root / config["collection"]["national_hydrography"]["river_geometry"]
display(validate_geometry(river_geometry, raise_on_error=False))

base_root = location_root / config["wflow"].get("base_model_root", "data/wflow/base")
domain_manifest = location_root / config["wflow"].get("domain_set_manifest", "data/wflow/domain_set.yaml")
configured_submodels = list(config["wflow"].get("domain_set", {}).get("submodels", []) or [])
if not configured_submodels and domain_manifest.exists():
    configured_submodels = list((yaml.safe_load(domain_manifest.read_text(encoding="utf-8")) or {}).get("submodels", []) or [])
if not configured_submodels:
    raise RuntimeError(f"No Wflow submodels found in config or generated manifest: {domain_manifest}")

handoff_artifact_qa = validate_wflow_sfincs_handoff_artifacts_current(
    config,
    location_root,
    submodels=configured_submodels,
    raise_on_error=False,
)
display(handoff_artifact_qa)
if not handoff_artifact_qa.empty and handoff_artifact_qa["status"].isin(["failed", "review_required"]).any():
    failed_handoff_artifact_qa = handoff_artifact_qa[handoff_artifact_qa["status"].isin(["failed", "review_required"])]
    raise RuntimeError(
        "Wflow-SFINCS handoff artifacts are stale. Re-run 04/a_build_coupled_model.ipynb before dynamic handoff.\n"
        + failed_handoff_artifact_qa.to_string(index=False)
    )

qa_rows = []
for submodel in configured_submodels:
    model_root = base_root / str(submodel["wflow_submodel_id"])
    if (model_root / "staticmaps.nc").exists():
        repair_wflow_staticmaps_nodata(model_root)
        report = validate_staticmaps(model_root, river_upa_km2=config["inland_coupling"]["discharge_forcing"].get("river_upa_km2"), raise_on_error=False)
        report.insert(0, "submodel_id", str(submodel["wflow_submodel_id"]))
        qa_rows.append(report)
    else:
        qa_rows.append(pd.DataFrame([{
            "submodel_id": str(submodel["wflow_submodel_id"]),
            "check": "staticmaps_exists",
            "status": "failed",
            "message": f"Missing staticmaps.nc: {model_root / 'staticmaps.nc'}",
        }]))
staticmap_qa = pd.concat(qa_rows, ignore_index=True) if qa_rows else pd.DataFrame()
display(staticmap_qa)
if not staticmap_qa.empty and staticmap_qa["status"].isin(["failed", "review_required"]).any():
    failed_staticmap_qa = staticmap_qa[staticmap_qa["status"].isin(["failed", "review_required"])]
    raise RuntimeError(
        "Wflow staticmap QA must pass before dynamic SFINCS handoff.\n"
        + failed_staticmap_qa.to_string(index=False)
    )

reservoir_readiness = write_wflow_reservoir_readiness(config, location_root, raise_on_error=False)
display(reservoir_readiness)
if not reservoir_readiness.empty and reservoir_readiness["status"].eq("failed").any():
    raise RuntimeError("Wflow reservoir QA must pass before dynamic SFINCS handoff.")


,check,status,message
0,rivwth,passed,valid=23604; unique=10199
1,rivdph,passed,valid=23604; unique=1418
2,stream_geo_source,passed,"source_columns=['rivwth_source', 'rivdph_sourc..."


,submodel_id,check,status,message
0,austin_p5u,wflow_gauges_sfincs_match_sources,passed,"expected=['austin_p5u_inflow_01', 'austin_p5u_..."
1,austin_p5u,wflow_domain_manifest_matches_sources,passed,"expected=['austin_p5u_inflow_01', 'austin_p5u_..."


,submodel_id,check,status,message
0,austin_p5u,nodata,passed,intmin_subcatchment=0; active_missing_ldd=0
1,austin_p5u,river_width,passed,valid_cells=7875; unique_values=566
2,austin_p5u,river_depth,passed,valid_cells=7875; unique_values=661
3,austin_p5u,land_slope,passed,missing_active_cells=0; max=0.496296
4,austin_p5u,river_slope,passed,missing_active_cells=0; max=0.0280704
5,austin_p5u,river_mask,passed,active_river_cells=7875; below_river_upa_thres...


,submodel_id,check,status,message
0,austin_p5u,reservoir_area_id,passed,"reservoir_ids=[1, 3, 5, 6]; area_cells=12898"
1,austin_p5u,reservoir_outlet_id,passed,"outlet_ids=[1, 3, 5, 6]; outlet_cells=4; missi..."
2,austin_p5u,reservoir_initial_depth,passed,valid_cells=4; min=3.65882; max=16.9408
3,austin_p5u,meta_reservoir_mean_outflow,passed,valid_cells=4; min=3.17324; max=9.09321
4,austin_p5u,reservoir_b,passed,valid_cells=4; min=0.0257133; max=0.363729
5,austin_p5u,reservoir_e,passed,valid_cells=4; min=2; max=2
6,austin_p5u,reservoir_rating_curve,passed,valid_cells=12898; min=-999; max=3
7,austin_p5u,reservoir_storage_curve,passed,valid_cells=12898; min=-999; max=1
8,austin_p5u,reservoir_outlets,passed,reservoir_ids=4; outlet_ids=4; missing_outlet_...
9,austin_p5u,reservoir_outlet_river_distance,passed,max_distance_m=0.0541136; too_far_ids=[]


### Collect Shared Warmup AORC Forcing


In [5]:
baseline_plan = plan_warmup(config, location_root)
display(baseline_plan)

collect_baseline_warmup = True
if collect_baseline_warmup:
    warmup_collection = collect_warmup(config, paths, force=rerun)
    display(pd.Series(warmup_collection, name="aorc_wflow_baseline_warmup"))

warmup_forcing = validate_warmup_forcing(
    config,
    location_root,
    event_id,
    reference_time=baseline_plan["baseline_reference_time"],
    raise_on_error=False,
)
display(warmup_forcing)
if warmup_forcing["status"].eq("failed").any():
    raise RuntimeError("Collect the shared AORC warmup baseline before running dynamic Wflow handoff.")

state_bootstrap = prepare_instates(config, location_root, force=rerun)
display(state_bootstrap)
if not state_bootstrap.empty and state_bootstrap["status"].eq("failed").any():
    raise RuntimeError("Create native Wflow instate/instates.nc before event replay.")

instate_readiness = validate_instates(config, location_root, raise_on_error=False)
display(instate_readiness)
reservoir_state_rows = []
for row in instate_readiness.itertuples(index=False):
    model_root = Path(row.instate).parents[1]
    if Path(row.instate).exists():
        report = validate_wflow_reservoir_states(model_root, required=True, raise_on_error=False)
        report.insert(0, "submodel_id", row.submodel_id)
        reservoir_state_rows.append(report)
reservoir_state_readiness = pd.concat(reservoir_state_rows, ignore_index=True) if reservoir_state_rows else pd.DataFrame()
display(reservoir_state_readiness)
if not instate_readiness.empty and instate_readiness["status"].eq("failed").any():
    raise RuntimeError("Create or promote native Wflow instate/instates.nc from the shared warmup baseline before event replay.")
if not reservoir_state_readiness.empty and reservoir_state_readiness["status"].eq("failed").any():
    raise RuntimeError("Promote a warmup-produced Wflow instate with reservoir_water_level before event replay.")


state_policy                                                 shared_baseline
baseline_id                                                     baseline_90d
baseline_reference_time                                  2020-11-14T00:00:00
warmup_days                                                             90.0
warmup_start                                             2020-08-16T00:00:00
warmup_end                                               2020-11-13T23:00:00
cold_state_workflow        /home/grahamhults/projects/Flood-RM/locations/...
warmup_baseline_root       /home/grahamhults/projects/Flood-RM/locations/...
warmup_precip              /home/grahamhults/projects/Flood-RM/locations/...
warmup_temp_pet            /home/grahamhults/projects/Flood-RM/locations/...
state_input                                              instate/instates.nc
state_output                                           outstate/outstates.nc
base_model_root            /home/grahamhults/projects/Flood-RM/locations/...

status                                                            collected
baseline_id                                                    baseline_90d
reference_time                                          2020-11-14T00:00:00
warmup_days                                                            90.0
warmup_start                                            2020-08-16T00:00:00
warmup_end                                              2020-11-13T23:00:00
source                                                                 AORC
source_nc                 /home/grahamhults/projects/Flood-RM/locations/...
precip_nc                 /home/grahamhults/projects/Flood-RM/locations/...
temp_pet_nc               /home/grahamhults/projects/Flood-RM/locations/...
temp_pet_provenance       {'source_nc': '/home/grahamhults/projects/Floo...
hydromt_wflow_contract        setup_precip_forcing + setup_temp_pet_forcing
Name: aorc_wflow_baseline_warmup, dtype: object

,file,variable,status,message
0,/home/grahamhults/projects/Flood-RM/locations/...,precip,passed,time_min=2020-08-16T00:00:00; time_max=2020-11...
1,/home/grahamhults/projects/Flood-RM/locations/...,temp,passed,time_min=2020-08-16T00:00:00; time_max=2020-11...


2026-06-29 11:11:00,985 - hydromt.model.model - model - INFO - Initializing wflow_sbm model from hydromt_wflow (v1.0.2).
2026-06-29 11:11:00,987 - hydromt.data_catalog.data_catalog - data_catalog - INFO - Parsing data catalog from /home/grahamhults/projects/Flood-RM/.venv/lib/python3.13/site-packages/hydromt_wflow/data/parameters_data.yml
2026-06-29 11:11:01,021 - hydromt.hydromt_wflow.wflow_base - wflow_base - INFO - Supported Wflow.jl version v1+
2026-06-29 11:11:01,023 - hydromt.hydromt_wflow.components.config - config - INFO - Reading model config file from /home/grahamhults/projects/Flood-RM/locations/austin/data/wflow/base/austin_p5u/wflow_sbm.toml.
2026-06-29 11:11:01,025 - hydromt.hydromt_wflow.components.config - config - INFO - Reading model config file from /home/grahamhults/projects/Flood-RM/locations/austin/data/wflow/base/austin_p5u/wflow_sbm.toml.
2026-06-29 11:11:01,941 - hydromt.hydromt_wflow.components.tables - tables - INFO - Reading model table files.
2026-06-29 11:

,submodel_id,instate,status,message
0,austin_p5u,/home/grahamhults/projects/Flood-RM/locations/...,prepared,native setup_cold_states timestamp=2020-08-16T...


,submodel_id,instate,status,message
0,austin_p5u,/home/grahamhults/projects/Flood-RM/locations/...,passed,ready


,submodel_id,check,status,message
0,austin_p5u,reservoir_water_level,passed,valid_cells=4; min=3.65882; max=16.9408


### Add req'd watershed meteorological variables


In [6]:
meteo_report = build_meteo(
    config,
    location_root,
    event_id,
    catalog_path=location_root / CATALOG_PATH,
    overwrite=rerun,
)
display(pd.Series(meteo_report, name="wflow_event_meteo_forcing"))


event_id                                                       design_0499
window_start                                           1998-10-15T17:00:00
window_end                                             1998-10-20T17:00:00
rainfall_source_nc       /home/grahamhults/projects/Flood-RM/locations/...
rainfall_scale_factor                                             1.484365
precip_path              /home/grahamhults/projects/Flood-RM/locations/...
temp_pet_path            /home/grahamhults/projects/Flood-RM/locations/...
precip_provenance        /home/grahamhults/projects/Flood-RM/locations/...
temp_pet_provenance      /home/grahamhults/projects/Flood-RM/locations/...
precip_written                                                        True
temp_pet_written                                                      True
Name: wflow_event_meteo_forcing, dtype: object

### Run and Create Wflow Startup File


In [7]:
run_dynamic_wflow = True

handoff_report = prepare_handoff(
    config,
    location_root,
    event_id,
    catalog_path=location_root / CATALOG_PATH,
    execute=run_dynamic_wflow,
)
display(handoff_report)


2026-06-29 11:11:08,796 - hydromt - log - INFO - HydroMT version: 1.3.1
2026-06-29 11:11:09,022 - hydromt.data_catalog.data_catalog - data_catalog - INFO - Parsing data catalog from /home/grahamhults/projects/Flood-RM/locations/austin/data/wflow/events/design_0499/_replay_data_catalog.yml
2026-06-29 11:11:09,050 - hydromt.model.model - model - INFO - Initializing wflow_sbm model from hydromt_wflow (v1.0.2).
2026-06-29 11:11:09,050 - hydromt.data_catalog.data_catalog - data_catalog - INFO - Parsing data catalog from /home/grahamhults/projects/Flood-RM/.venv/lib/python3.13/site-packages/hydromt_wflow/data/parameters_data.yml
2026-06-29 11:11:09,065 - hydromt.hydromt_wflow.wflow_base - wflow_base - INFO - Supported Wflow.jl version v1+
2026-06-29 11:11:09,065 - hydromt.hydromt_wflow.components.config - config - INFO - Reading model config file from /home/grahamhults/projects/Flood-RM/locations/austin/data/wflow/base/austin_p5u/wflow_sbm.toml.
2026-06-29 11:11:09,077 - hydromt - log - INFO

┌ Warning: 'header' is not recognized as a valid field of the [input] section in the TOML, this will be ignored.
└ @ Wflow ~/.julia/packages/Wflow/mJ7Ug/src/config_init.jl:81
┌ Warning: 'params' is not recognized as a valid field of the [input] section in the TOML, this will be ignored.
└ @ Wflow ~/.julia/packages/Wflow/mJ7Ug/src/config_init.jl:81
[ Info: Wflow version v1.0.2
[ Info: Initialize model variables for model type sbm.
┌ Info: Cyclic parameters are provided by
└ /home/grahamhults/projects/Flood-RM/locations/austin/data/wflow/events/design_0499/austin_p5u/staticmaps.nc.
┌ Info: Forcing parameters are provided by
└ /home/grahamhults/projects/Flood-RM/locations/austin/data/wflow/events/design_0499/austin_p5u/inmaps-event.nc.
┌ Info: Set atmosphere_water__precipitation_volume_flux using netCDF variable
└ precip as forcing parameter.
┌ Info: Set atmosphere_air__temperature using netCDF variable temp as forcing
└ parameter.
┌ Info: Set land_surface_water__potential_evaporation_vol

,event_id,submodel_id,window_start,window_end,update_command,resolved_update_command,hydromt_runner_status,hydromt_runner_issue,run_command,run_output_dir,status,sfincs_discharge_forcing,sfincs_discharge_written,sfincs_discharge_source,dynamic_handoff_acceptance,acceptance_status
0,design_0499,austin_p5u,1998-10-15T17:00:00,1998-10-20T17:00:00,hydromt update wflow_sbm /home/grahamhults/pro...,/home/grahamhults/projects/Flood-RM/.venv/bin/...,project_venv,,wflow_cli /home/grahamhults/projects/Flood-RM/...,/home/grahamhults/projects/Flood-RM/locations/...,completed,/home/grahamhults/projects/Flood-RM/locations/...,True,wflow_dynamic,/home/grahamhults/projects/Flood-RM/locations/...,accepted


### Results


In [8]:
acceptance_path = location_root / handoff_plan["dynamic_handoff_acceptance"]
if acceptance_path.exists():
    display(pd.read_json(acceptance_path, typ="series"))
else:
    print("Dynamic handoff planned but not executed yet:", acceptance_path)


event_id                                                  design_0499
status                                                       accepted
discharge_source                                        wflow_dynamic
discharge_nc        /home/grahamhults/projects/Flood-RM/locations/...
checks              [{'check': 'event_peak', 'status': 'passed', '...
metadata            {'warmup_state_plan': {'state_policy': 'shared...
dtype: object